In [1]:
pip --version

pip 26.2.1 from C:\Users\PGCP-AI.STUDENTSDC\AppData\Local\Programs\Python\Python314\Lib\site-packages\pip (python 3.14)

Note: you may need to restart the kernel to use updated packages.


In [ ]:
from sentence_transformers import SentenceTransformer
import pandas as pd
import numpy as np
from tqdm import tqdm

In [ ]:
model = SentenceTransformer('BAAI/bge-small-en-v1.5')
# model = SentenceTransformer('sentance-transfomrer/all-MiniLM-L6-v2')

In [ ]:
sentences = [
    'He batted well today in the cricket match.',
    'Tommorrow will be Wednesday',
    'She played excellently yesterday with many goals',
    'Today will be the event',
    'He performed well in that game'
]

In [ ]:
embedding = model.encode(sentences)

In [ ]:
similarities = model.similarity(embedding,embedding)
similarities.shape

In [ ]:
similarities

In [ ]:
df = pd.read_csv('/content/titles.csv')
df.head()

In [ ]:
df_ind_movies = df[(df['production_countries'].str.contains('IN')) &
    (df['type'] == 'MOVIE') & (df['release_year'] >= 1980)]
df_ind_movies.shape

In [ ]:
all_desc = list(df_ind_movies['description'])
embeds = []
for r in tqdm(all_desc):
    embd = model.encode(r)
    embeds.append(embd)
out_embs = np.array(embeds)
out_embs.shape

In [ ]:
embeds_df = pd.DataFrame(out_embs)
embeds_df

In [ ]:
movie_indx = list(df_ind_movies['title']+'-'+df_ind_movies['release_year'].astype(str))
embeds_df.index = movie_indx
embeds_df.head()

In [ ]:
embeds_df.to_csv('embed_movie')

In [ ]:
# Movie Search by Description

In [ ]:
def find_movie(query):
  qy_emb = model.encode([query])
  sims = []
  for e in tqdm(out_embs):
    sims.append(model.similarity(qy_emb,e).numpy()[0])
  df_ind = df_ind_movies[['id','title','description','release_year']].copy()
  df_ind['sims'] = np.array(sims)
  return df_ind.sort_values('sims',ascending=False).head(10)

In [ ]:
query = input('Enter Desription :-')
find_movie(query)